# Data Cleaning and Feature Preparation

## The Dataset: Titanic Passenger Records

**1. Load a Dataset Directly from a GitHub Repository**

In [ ]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/datasciencedojo/"
    "datasets/master/titanic.csv"
)

df = pd.read_csv(url)

print(df.shape)    # (891, 12)
print(df.dtypes)

(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


**2. Load a Dataset Locally**

If the dataset is already downloaded to your machine:

In [2]:
import pandas as pd

path = "../data/titanic.csv"
df = pd.read_csv(path)

print(df.shape)    # (891, 12)
print(df.dtypes)

(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


## Handling Missing Data

### Detecting missing values

In [3]:
missing = df.isnull().sum()
print(missing[missing > 0])

Age         177
Cabin       687
Embarked      2
dtype: int64


In [4]:
missing_pct = df.isnull().mean() * 100
print(missing_pct[missing_pct > 0].round(1))

Age         19.9
Cabin       77.1
Embarked     0.2
dtype: float64


### Dropping rows and columns

In [5]:
# Cabin is 77% missing — drop the column entirely
df = df.drop(columns=["Cabin"])
print(df.shape)   # (891, 11)

(891, 11)


In [6]:
df = df.dropna(subset=["Embarked"])
print(df.shape)   # (889, 11)

(889, 11)


### Mean and median imputation

In [7]:
# Age distribution is right-skewed — use median imputation
age_median = df["Age"].median()
print(f"Median age: {age_median}")   # 28.0

df["Age"] = df["Age"].fillna(age_median)
print(df["Age"].isnull().sum())      # 0

Median age: 28.0
0


In [8]:
df["Age"] = df.groupby("Pclass")["Age"].transform(
    lambda x: x.fillna(x.median())
)
print(df.groupby("Pclass")["Age"].median().round(1))

Pclass
1    35.0
2    28.0
3    28.0
Name: Age, dtype: float64


### Forward fill and backward fill

In [9]:
# Illustrative example on a small Series
import pandas as pd
s = pd.Series([1.0, None, None, 4.0, None])
print(s.ffill())   # [1, 1, 1, 4, 4]
print(s.bfill())   # [1, 4, 4, 4, NaN]

0    1.0
1    1.0
2    1.0
3    4.0
4    4.0
dtype: float64
0    1.0
1    4.0
2    4.0
3    4.0
4    NaN
dtype: float64


### Missing value indicators

In [10]:
df["Age_missing"] = df["Age"].isnull().astype(int)
print(df["Age_missing"].value_counts())

Age_missing
0    889
Name: count, dtype: int64


## Outlier Detection

In [11]:
print(df[["Age", "Fare", "SibSp", "Parch"]].describe().round(2))

          Age    Fare   SibSp   Parch
count  889.00  889.00  889.00  889.00
mean    29.32   32.10    0.52    0.38
std     12.98   49.70    1.10    0.81
min      0.42    0.00    0.00    0.00
25%     22.00    7.90    0.00    0.00
50%     28.00   14.45    0.00    0.00
75%     35.00   31.00    1.00    0.00
max     80.00  512.33    8.00    6.00


### The IQR method

In [12]:
Q1   = df["Fare"].quantile(0.25)
Q3   = df["Fare"].quantile(0.75)
IQR  = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["Fare"] < lower) | (df["Fare"] > upper)]
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
print(f"Outliers detected: {len(outliers)}")

IQR bounds: [-26.76, 65.66]
Outliers detected: 114


### Z-score method

In [13]:
import numpy as np

fare_mean = df["Fare"].mean()
fare_std  = df["Fare"].std()

df["Fare_zscore"] = (df["Fare"] - fare_mean) / fare_std

extreme = df[df["Fare_zscore"].abs() > 3]
print(f"Extreme fares (|Z| > 3): {len(extreme)}")
print(extreme["Fare"].describe().round(2))

Extreme fares (|Z| > 3): 20
count     20.00
mean     279.31
std      102.35
min      211.34
25%      226.09
50%      247.52
75%      263.00
max      512.33
Name: Fare, dtype: float64


### Capping (Winsorisation)

In [14]:
upper_cap = df["Fare"].quantile(0.99)
df["Fare_capped"] = df["Fare"].clip(upper=upper_cap)

print(f"Original max Fare:  {df['Fare'].max():.2f}")
print(f"Capped max Fare:    {df['Fare_capped'].max():.2f}")

Original max Fare:  512.33
Capped max Fare:    249.30


## Data Transformation

### Min-Max scaling

In [15]:
def minmax_scale(series):
    return (series - series.min()) / (series.max() - series.min())

df["Age_scaled"]  = minmax_scale(df["Age"])
df["Fare_scaled"] = minmax_scale(df["Fare"])

print(df[["Age_scaled", "Fare_scaled"]].describe().round(3))

       Age_scaled  Fare_scaled
count     889.000      889.000
mean        0.363        0.063
std         0.163        0.097
min         0.000        0.000
25%         0.271        0.015
50%         0.347        0.028
75%         0.435        0.061
max         1.000        1.000


### Standardisation (Z-score scaling)

In [16]:
def standardise(series):
    return (series - series.mean()) / series.std()

df["Age_z"]  = standardise(df["Age"])
df["Fare_z"] = standardise(df["Fare"])

print(df[["Age_z", "Fare_z"]].agg(["mean", "std"]).round(6))

      Age_z  Fare_z
mean    0.0     0.0
std     1.0     1.0


### Robust scaling

In [17]:
def robust_scale(series):
    median = series.median()
    iqr    = series.quantile(0.75) - series.quantile(0.25)
    return (series - median) / iqr

df["Fare_robust"] = robust_scale(df["Fare"])
print(df["Fare_robust"].describe().round(3))

count    889.000
mean       0.764
std        2.151
min       -0.626
25%       -0.284
50%        0.000
75%        0.716
max       21.549
Name: Fare_robust, dtype: float64


### Log transformation

In [18]:
df["Fare_log"] = np.log1p(df["Fare"])   # log(1 + x) — safe for zero fares

print(f"Fare skewness (original): {df['Fare'].skew():.2f}")
print(f"Fare skewness (log):      {df['Fare_log'].skew():.2f}")

Fare skewness (original): 4.80
Fare skewness (log):      0.40


## Encoding Categorical Variables

### Label encoding

In [19]:
# Pclass is ordinal: 1 (first) > 2 (second) > 3 (third)
# It is already integer-coded in this dataset, so no conversion needed.

# Survived is binary — label encoding is correct
df["Survived_encoded"] = df["Survived"]   # already 0/1

# Manual label encoding for a hypothetical ordinal column
size_map = {"small": 0, "medium": 1, "large": 2}
# df["size_encoded"] = df["size"].map(size_map)

### One-hot encoding

In [21]:
# Encode Sex and Embarked
df_encoded = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)

# drop_first=True drops one dummy per variable to avoid perfect multicollinearity
new_cols = [c for c in df_encoded.columns if "Sex_" in c or "Embarked_" in c]
print(new_cols)
print(df_encoded[new_cols].head(3))

['Sex_male', 'Embarked_Q', 'Embarked_S']
   Sex_male  Embarked_Q  Embarked_S
0      True       False        True
1     False       False       False
2     False       False        True


### Ordinal encoding

In [22]:
# Map Pclass to an explicit ordinal: 1st=3, 2nd=2, 3rd=1 (reverse: higher value = higher class)
pclass_order = {1: 3, 2: 2, 3: 1}
df["Pclass_ordinal"] = df["Pclass"].map(pclass_order)
print(df[["Pclass", "Pclass_ordinal"]].value_counts().sort_index())

Pclass  Pclass_ordinal
1       3                 214
2       2                 184
3       1                 491
Name: count, dtype: int64


### Frequency encoding

In [23]:
# Encode ticket prefix frequency (high-cardinality column)
df["Ticket_prefix"] = df["Ticket"].str.extract(r"([A-Za-z]+)", expand=False).fillna("NONE")
freq_map = df["Ticket_prefix"].value_counts()
df["Ticket_prefix_freq"] = df["Ticket_prefix"].map(freq_map)
print(df[["Ticket_prefix", "Ticket_prefix_freq"]].head(5))

  Ticket_prefix  Ticket_prefix_freq
0             A                  29
1            PC                  60
2          STON                  18
3          NONE                 659
4          NONE                 659


## Feature Engineering

### Extracting titles from names

In [24]:
# Extract title from Name column
df["Title"] = df["Name"].str.extract(r",\s*([A-Za-z]+)\.")
print(df["Title"].value_counts())

Title
Mr          517
Miss        181
Mrs         124
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Jonkheer      1
Name: count, dtype: int64


In [25]:
common_titles = ["Mr", "Miss", "Mrs", "Master"]
df["Title_grouped"] = df["Title"].where(df["Title"].isin(common_titles), other="Rare")
print(df["Title_grouped"].value_counts())

Title_grouped
Mr        517
Miss      181
Mrs       124
Master     40
Rare       27
Name: count, dtype: int64


### Family size

In [26]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1   # +1 for the passenger themselves

# Further bin into categorical groups
def family_group(n):
    if n == 1:
        return "alone"
    elif n <= 4:
        return "small"
    else:
        return "large"

df["FamilyGroup"] = df["FamilySize"].apply(family_group)
print(df["FamilyGroup"].value_counts())

FamilyGroup
alone    535
small    292
large     62
Name: count, dtype: int64


### Binning continuous variables

In [27]:
# Equal-width age bins
df["Age_band"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)
print(df["Age_band"].value_counts().sort_index())

Age_band
Child          69
Teen           70
YoungAdult    535
Adult         194
Senior         21
Name: count, dtype: int64


### Interaction features

In [28]:
# Interaction: Sex × Pclass
df["Sex_Pclass"] = df["Sex"].astype(str) + "_" + df["Pclass"].astype(str)
print(df.groupby("Sex_Pclass")["Survived"].mean().round(3))

Sex_Pclass
female_1    0.967
female_2    0.921
female_3    0.500
male_1      0.369
male_2      0.157
male_3      0.135
Name: Survived, dtype: float64


### Is the passenger travelling alone?

In [29]:
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
print(df.groupby("IsAlone")["Survived"].mean().round(3))

IsAlone
0    0.506
1    0.301
Name: Survived, dtype: float64


### Fare per person

In [30]:
df["Fare_per_person"] = df["Fare"] / df["FamilySize"]
print(df[["Fare", "FamilySize", "Fare_per_person"]].head(5).round(2))

    Fare  FamilySize  Fare_per_person
0   7.25           2             3.62
1  71.28           2            35.64
2   7.92           1             7.92
3  53.10           2            26.55
4   8.05           1             8.05
